
# Week 2 - Mục 2.3 Validation Notebook
## Kiểm tra `compute_fiedler_vector()` cho cả 2 cách tính Laplacian

Notebook này kiểm tra song song hai biến thể:

1. **Unnormalized Laplacian**: `L = D - A`
2. **Normalized Laplacian**: `L = I - D^{-1/2} A D^{-1/2}`

Các điều kiện kiểm tra:

### A. Với **unnormalized Laplacian**
- **Fiedler vector trực giao với** `[1, 1, ..., 1]`
- Điều kiện: `abs(dot(fiedler, 1)) < 1e-4`
- **Weights** sau khi lấy `abs(fiedler)` và normalize có tổng bằng 1
- Điều kiện: `abs(weights.sum() - 1.0) < 1e-6`

### B. Với **normalized Laplacian**
- Eigenvector nhỏ nhất của `L_sym` là `sqrt(D) * 1`, nên **Fiedler vector phải trực giao với `sqrt(degree)`**, không phải luôn trực giao với vector toàn 1
- Điều kiện: `abs(dot(fiedler, sqrt_degree)) < 1e-4`
- **Weights** sau khi lấy `abs(fiedler)` và normalize có tổng bằng 1
- Điều kiện: `abs(weights.sum() - 1.0) < 1e-6`

> Ghi chú: notebook vẫn in thêm `dot(fiedler, 1)` cho trường hợp normalized như một **diagnostic**, nhưng đó không phải acceptance criterion chính.


In [35]:
from pathlib import Path
import sys

repo_root = Path("/Users/tienesng06/Desktop/ACIVS_ThayBach")   # sửa đúng path repo của bạn

assert repo_root.exists(), f"Repo path does not exist: {repo_root}"
assert (repo_root / "src" / "models").exists(), "src/models not found in repo"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("repo_root:", repo_root)

repo_root: /Users/tienesng06/Desktop/ACIVS_ThayBach


In [36]:

USING_PROJECT_IMPL = False
PROJECT_SUPPORTS_NORMALIZED = False

project_compute_fiedler_vector = None
project_compute_fiedler_magnitude_weights = None

try:
    from src.models.fiedler import (
        compute_fiedler_vector as project_compute_fiedler_vector,
        compute_fiedler_magnitude_weights as project_compute_fiedler_magnitude_weights,
    )
    USING_PROJECT_IMPL = True

    sig_f = inspect.signature(project_compute_fiedler_vector)
    sig_w = inspect.signature(project_compute_fiedler_magnitude_weights)
    PROJECT_SUPPORTS_NORMALIZED = (
        "normalized" in sig_f.parameters and "normalized" in sig_w.parameters
    )

    print("Imported functions from src.models.fiedler")
    print("Project supports normalized flag:", PROJECT_SUPPORTS_NORMALIZED)
except Exception as e:
    print("Could not import project implementation:", repr(e))
    print("Notebook will use local reference implementation.")


def ref_compute_graph_laplacian(
    A: torch.Tensor,
    normalized: bool = True,
    eps: float = 1e-12,
) -> torch.Tensor:
    A = A.to(torch.float64)
    A = 0.5 * (A + A.T)
    degree = A.sum(dim=1)

    if normalized:
        inv_sqrt_degree = torch.rsqrt(torch.clamp(degree, min=eps))
        D_inv_sqrt = torch.diag(inv_sqrt_degree)
        I = torch.eye(A.shape[0], dtype=A.dtype, device=A.device)
        L = I - D_inv_sqrt @ A @ D_inv_sqrt
    else:
        D = torch.diag(degree)
        L = D - A

    return 0.5 * (L + L.T)


def ref_compute_fiedler_vector(
    A: torch.Tensor,
    normalized: bool = True,
    eps: float = 1e-12,
) -> torch.Tensor:
    L = ref_compute_graph_laplacian(A, normalized=normalized, eps=eps)
    eigenvalues, eigenvectors = torch.linalg.eigh(L)
    idx = torch.argsort(eigenvalues)
    eigenvectors = eigenvectors[:, idx]
    return eigenvectors[:, 1]


def ref_compute_fiedler_magnitude_weights(
    A: torch.Tensor,
    normalized: bool = True,
    eps: float = 1e-12,
) -> torch.Tensor:
    f = ref_compute_fiedler_vector(A, normalized=normalized, eps=eps)
    w = torch.abs(f)
    return w / torch.clamp(w.sum(), min=eps)


def get_fiedler_vector(A: torch.Tensor, normalized: bool) -> tuple[torch.Tensor, str]:
    if USING_PROJECT_IMPL:
        if PROJECT_SUPPORTS_NORMALIZED:
            return project_compute_fiedler_vector(A, normalized=normalized), "project"
        if not normalized:
            return project_compute_fiedler_vector(A), "project (unnormalized-only)"
    return ref_compute_fiedler_vector(A, normalized=normalized), "reference"


def get_fiedler_weights(A: torch.Tensor, normalized: bool) -> tuple[torch.Tensor, str]:
    if USING_PROJECT_IMPL:
        if PROJECT_SUPPORTS_NORMALIZED:
            return project_compute_fiedler_magnitude_weights(A, normalized=normalized), "project"
        if not normalized:
            return project_compute_fiedler_magnitude_weights(A), "project (unnormalized-only)"
    return ref_compute_fiedler_magnitude_weights(A, normalized=normalized), "reference"


Imported functions from src.models.fiedler
Project supports normalized flag: True



## 1) Tạo affinity matrix đối xứng để test

Dùng synthetic graph có 2 cụm band:
- trong cùng cụm: affinity cao
- khác cụm: affinity thấp

Đây là case trực quan nhất để kiểm tra Fiedler vector.


In [37]:

def make_two_cluster_affinity(
    cluster_sizes=(6, 7),
    intra=0.9,
    inter=0.1,
    self_loop=1.0,
    dtype=torch.float64,
):
    B = sum(cluster_sizes)
    A = torch.full((B, B), inter, dtype=dtype)

    start = 0
    for size in cluster_sizes:
        end = start + size
        A[start:end, start:end] = intra
        start = end

    A.fill_diagonal_(self_loop)
    A = 0.5 * (A + A.T)
    return A

A = make_two_cluster_affinity()
print("A shape:", tuple(A.shape))
A


A shape: (13, 13)


tensor([[1.000000, 0.900000, 0.900000, 0.900000, 0.900000, 0.900000, 0.100000,
         0.100000, 0.100000, 0.100000, 0.100000, 0.100000, 0.100000],
        [0.900000, 1.000000, 0.900000, 0.900000, 0.900000, 0.900000, 0.100000,
         0.100000, 0.100000, 0.100000, 0.100000, 0.100000, 0.100000],
        [0.900000, 0.900000, 1.000000, 0.900000, 0.900000, 0.900000, 0.100000,
         0.100000, 0.100000, 0.100000, 0.100000, 0.100000, 0.100000],
        [0.900000, 0.900000, 0.900000, 1.000000, 0.900000, 0.900000, 0.100000,
         0.100000, 0.100000, 0.100000, 0.100000, 0.100000, 0.100000],
        [0.900000, 0.900000, 0.900000, 0.900000, 1.000000, 0.900000, 0.100000,
         0.100000, 0.100000, 0.100000, 0.100000, 0.100000, 0.100000],
        [0.900000, 0.900000, 0.900000, 0.900000, 0.900000, 1.000000, 0.100000,
         0.100000, 0.100000, 0.100000, 0.100000, 0.100000, 0.100000],
        [0.100000, 0.100000, 0.100000, 0.100000, 0.100000, 0.100000, 1.000000,
         0.900000, 0.900000


## 2) Hàm kiểm tra cho từng biến thể

- **Unnormalized**: kiểm tra `dot(fiedler, 1)`
- **Normalized**: kiểm tra `dot(fiedler, sqrt_degree)`
- Cả hai: kiểm tra `weights.sum()`


In [38]:

def validate_fiedler_case(
    A: torch.Tensor,
    normalized: bool,
    ortho_tol: float = 1e-4,
    sum_tol: float = 1e-6,
):
    fiedler_vec, source_f = get_fiedler_vector(A, normalized=normalized)
    weights, source_w = get_fiedler_weights(A, normalized=normalized)

    fiedler_vec = fiedler_vec.to(torch.float64)
    weights = weights.to(torch.float64)
    degree = A.sum(dim=1).to(torch.float64)
    ones = torch.ones_like(fiedler_vec)
    sqrt_degree = torch.sqrt(degree)

    dot_with_ones = torch.dot(fiedler_vec, ones).abs().item()
    dot_with_sqrt_degree = torch.dot(fiedler_vec, sqrt_degree).abs().item()
    weight_sum = weights.sum().item()

    mode_name = "normalized" if normalized else "unnormalized"

    if normalized:
        ortho_metric_name = "|dot(fiedler, sqrt_degree)|"
        ortho_metric_value = dot_with_sqrt_degree
    else:
        ortho_metric_name = "|dot(fiedler, 1)|"
        ortho_metric_value = dot_with_ones

    return {
        "mode": mode_name,
        "source_fiedler": source_f,
        "source_weights": source_w,
        "fiedler_vec": fiedler_vec,
        "weights": weights,
        "dot_with_ones": dot_with_ones,
        "dot_with_sqrt_degree": dot_with_sqrt_degree,
        "ortho_metric_name": ortho_metric_name,
        "ortho_metric_value": ortho_metric_value,
        "weight_sum": weight_sum,
        "ortho_pass": ortho_metric_value < ortho_tol,
        "sum_pass": abs(weight_sum - 1.0) < sum_tol,
    }


ORTHO_TOL = 1e-4
SUM_TOL = 1e-6



## 3) Chạy kiểm tra chính cho cả hai cách tính


In [39]:

results_main = [
    validate_fiedler_case(A, normalized=False, ortho_tol=ORTHO_TOL, sum_tol=SUM_TOL),
    validate_fiedler_case(A, normalized=True, ortho_tol=ORTHO_TOL, sum_tol=SUM_TOL),
]

for r in results_main:
    print("=" * 70)
    print(f"Mode: {r['mode']}")
    print(f"Fiedler source: {r['source_fiedler']}")
    print(f"Weights source: {r['source_weights']}")
    print(r['ortho_metric_name'], "=", f"{r['ortho_metric_value']:.12f}")
    print("|dot(fiedler, 1)| =", f"{r['dot_with_ones']:.12f}")
    print("|dot(fiedler, sqrt_degree)| =", f"{r['dot_with_sqrt_degree']:.12f}")
    print("weights.sum() =", f"{r['weight_sum']:.12f}")
    print("orthogonality pass:", r['ortho_pass'])
    print("weight-sum pass:", r['sum_pass'])
    print("\nFiedler vector:")
    print(r['fiedler_vec'])
    print("\nWeights:")
    print(r['weights'])
    print()

summary_rows = []
for r in results_main:
    summary_rows.append({
        "mode": r["mode"],
        "criterion": r["ortho_metric_name"],
        "criterion_value": r["ortho_metric_value"],
        "weights_sum": r["weight_sum"],
        "orthogonality_pass": r["ortho_pass"],
        "sum_pass": r["sum_pass"],
    })

if pd is not None:
    display(pd.DataFrame(summary_rows))
else:
    print(summary_rows)


Mode: unnormalized
Fiedler source: project
Weights source: project
|dot(fiedler, 1)| = 0.000000983477
|dot(fiedler, 1)| = 0.000000983477
|dot(fiedler, sqrt_degree)| = 0.279991286205
weights.sum() = 0.999999962747
orthogonality pass: True
weight-sum pass: True

Fiedler vector:
tensor([ 0.299572,  0.299572,  0.299572,  0.299572,  0.299572,  0.299572,
        -0.256776, -0.256776, -0.256776, -0.256776, -0.256776, -0.256776,
        -0.256776], dtype=torch.float64)

Weights:
tensor([0.083333, 0.083333, 0.083333, 0.083333, 0.083333, 0.083333, 0.071429,
        0.071429, 0.071429, 0.071429, 0.071429, 0.071429, 0.071429],
       dtype=torch.float64)

Mode: normalized
Fiedler source: project
Weights source: project
|dot(fiedler, sqrt_degree)| = 0.000008796510
|dot(fiedler, 1)| = 0.108735740185
|dot(fiedler, sqrt_degree)| = 0.000008796510
weights.sum() = 0.999999985099
orthogonality pass: True
weight-sum pass: True

Fiedler vector:
tensor([-0.307800, -0.307800, -0.307800, -0.307800, -0.307800, 

,mode,criterion,criterion_value,weights_sum,orthogonality_pass,sum_pass
0,unnormalized,"|dot(fiedler, 1)|",9.834766e-07,1.0,True,True
1,normalized,"|dot(fiedler, sqrt_degree)|",8.796510e-06,1.0,True,True


In [40]:

for r in results_main:
    assert r["ortho_pass"], (
        f"FAILED [{r['mode']}]: {r['ortho_metric_name']} = "
        f"{r['ortho_metric_value']:.10f} >= {ORTHO_TOL}"
    )
    assert r["sum_pass"], (
        f"FAILED [{r['mode']}]: weights.sum() = {r['weight_sum']:.10f} "
        f"is not within {SUM_TOL} of 1.0"
    )

print("PASS")
print(f"- Unnormalized: |dot(fiedler, 1)| < {ORTHO_TOL}")
print(f"- Normalized: |dot(fiedler, sqrt_degree)| < {ORTHO_TOL}")
print(f"- Both modes: |weights.sum() - 1| < {SUM_TOL}")


PASS
- Unnormalized: |dot(fiedler, 1)| < 0.0001
- Normalized: |dot(fiedler, sqrt_degree)| < 0.0001
- Both modes: |weights.sum() - 1| < 1e-06



## 4) So sánh hai cách tính trên cùng một graph

Cell này không yêu cầu hai cách phải cho ra cùng vector hoặc cùng weights.
Nó chỉ giúp bạn nhìn thấy mức độ khác nhau giữa hai biến thể trên cùng một affinity graph.


In [41]:

unnorm = results_main[0]
norm = results_main[1]

l1_diff = torch.sum(torch.abs(unnorm["weights"] - norm["weights"])).item()
cos_sim = torch.nn.functional.cosine_similarity(
    unnorm["weights"].unsqueeze(0),
    norm["weights"].unsqueeze(0),
).item()

print("L1 difference between weight vectors:", f"{l1_diff:.12f}")
print("Cosine similarity between weight vectors:", f"{cos_sim:.12f}")


L1 difference between weight vectors: 0.030332140625
Cosine similarity between weight vectors: 0.999545139634



## 5) Stress test trên nhiều graph ngẫu nhiên đối xứng

- Với **unnormalized**: kiểm tra `dot(fiedler, 1)`
- Với **normalized**: kiểm tra `dot(fiedler, sqrt_degree)`
- Cả hai: kiểm tra `weights.sum()`


In [43]:

def make_random_symmetric_affinity(B=13, seed=0, dtype=torch.float64):
    g = torch.Generator().manual_seed(seed)
    X = torch.rand((B, B), generator=g, dtype=dtype)
    A = 0.5 * (X + X.T)
    A.fill_diagonal_(1.0)
    return A

stress_rows = []
for seed in range(10):
    A_rand = make_random_symmetric_affinity(B=13, seed=seed)
    for normalized in (False, True):
        r = validate_fiedler_case(A_rand, normalized=normalized, ortho_tol=ORTHO_TOL, sum_tol=SUM_TOL)
        stress_rows.append({
            "seed": seed,
            "mode": r["mode"],
            "criterion": r["ortho_metric_name"],
            "criterion_value": r["ortho_metric_value"],
            "weights_sum": r["weight_sum"],
            "orthogonality_pass": r["ortho_pass"],
            "sum_pass": r["sum_pass"],
        })

if pd is not None:
    stress_df = pd.DataFrame(stress_rows)
    display(stress_df)
    print("\nPass counts by mode:")
    display(stress_df.groupby("mode")[["orthogonality_pass", "sum_pass"]].sum())
else:
    print(stress_rows)


,seed,mode,criterion,criterion_value,weights_sum,orthogonality_pass,sum_pass
0,0,unnormalized,"|dot(fiedler, 1)|",1.057051e-07,1.0,True,True
1,0,normalized,"|dot(fiedler, sqrt_degree)|",3.286415e-07,1.0,True,True
2,1,unnormalized,"|dot(fiedler, 1)|",4.079193e-07,1.0,True,True
3,1,normalized,"|dot(fiedler, sqrt_degree)|",4.928462e-07,1.0,True,True
4,2,unnormalized,"|dot(fiedler, 1)|",3.832392e-07,1.0,True,True
5,2,normalized,"|dot(fiedler, sqrt_degree)|",6.807573e-08,1.0,True,True
6,3,unnormalized,"|dot(fiedler, 1)|",5.264301e-07,1.0,True,True
7,3,normalized,"|dot(fiedler, sqrt_degree)|",1.199939e-06,1.0,True,True
8,4,unnormalized,"|dot(fiedler, 1)|",2.430752e-07,1.0,True,True
9,4,normalized,"|dot(fiedler, sqrt_degree)|",1.159438e-06,1.0,True,True



Pass counts by mode:


,orthogonality_pass,sum_pass
mode,,
normalized,10,10
unnormalized,10,10
